# 07 · Snapshots (SCD Type 2)

The source system **overwrites** customer addresses and order statuses --
history is lost upstream. Snapshots record every version with validity
intervals:

| column | meaning |
|---|---|
| `dbt_valid_from` | when this version became true |
| `dbt_valid_to` | when it stopped (NULL = current) |
| `dbt_scd_id` | surrogate key of (row, version) |

Companion reading: [docs/10](../docs/10_snapshots.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Snapshot the current state (idempotent if nothing changed)

In [ ]:
res = run_dbt(["snapshot"])
assert res.success

## 2. Advance business time one day (mutations included), snapshot again

In [ ]:
next_day = int(pd.read_sql("select max(_batch_day) + 1 as d from raw.raw_orders", engine)["d"][0])
load_day("--day", str(next_day))

res = run_dbt(["snapshot"])
assert res.success

## 3. Customers who moved: multiple versions, seamless intervals

In [ ]:
%%sql
select id, city, address,
       date_trunc('second', dbt_valid_from) as valid_from,
       date_trunc('second', dbt_valid_to)   as valid_to
from dev_snapshots.snap_customers
where id in (
    select id from dev_snapshots.snap_customers
    group by id having count(*) > 1
)
order by id, dbt_valid_from
limit 12

`snap_customers` uses the **timestamp** strategy: dbt trusted the source's
`updated_at` to detect changes, and `dbt_valid_from` of each new version is
that `updated_at`.

## 4. Order lifecycle, reconstructed by the **check** strategy

`snap_order_status` compares `check_cols: [status]` value-by-value instead
of trusting a timestamp:

In [ ]:
%%sql
select id, status,
       date_trunc('second', dbt_valid_from) as valid_from,
       date_trunc('second', dbt_valid_to)   as valid_to
from dev_snapshots.snap_order_status
where id in (
    select id from dev_snapshots.snap_order_status
    group by id having count(*) >= 3
)
order by id, dbt_valid_from
limit 12

## 5. The two canonical SCD-2 queries

In [ ]:
%%sql
-- CURRENT state: dbt_valid_to is null
select count(*) as current_customers
from dev_snapshots.snap_customers
where dbt_valid_to is null

In [ ]:
# POINT-IN-TIME: what did we believe at a past moment?
as_of = pd.read_sql(
    "select min(dbt_valid_to) + interval '1 second' as t from dev_snapshots.snap_customers",
    engine)["t"][0]
print(f"as of {as_of}:")

pd.read_sql(f"""
    select count(*) as customers_as_of_then,
           count(*) filter (where dbt_valid_to is not null) as later_superseded
    from dev_snapshots.snap_customers
    where dbt_valid_from <= '{as_of}'
      and (dbt_valid_to > '{as_of}' or dbt_valid_to is null)
""", engine)

## 6. Operational notes

* Snapshot **sources**, as early as possible -- history not captured now is
  gone forever. Both snapshots here read `source()`, not models.
* Snapshots are append-only state: `--full-refresh` does not rebuild them
  (by design), so treat the snapshot schema as production data even in dev.
* timestamp strategy: cheap, needs a trustworthy `updated_at`.
  check strategy: no timestamp needed, pays a column-compare per run.
* `dbt build` runs snapshots in DAG order too; scheduling snapshots more
  often than the source changes is free (no-ops).